In [1]:
!pip install langchain langchain-core langchain-community pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.7 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-groq]


In [20]:
from getpass import getpass
import os
import requests
from langchain_core.tools import tool
from langchain.messages import HumanMessage, ToolMessage
from langchain_groq import ChatGroq

os.environ['GROQ_API_KEY'] = getpass("Enter Groq API Key : ")
llm = ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=0
)
print(llm.invoke("Hello, introduce yourself").content)

Enter Groq API Key : ··········
Hello! I’m ChatGPT, a large language model created by OpenAI. I’m here to help answer questions, brainstorm ideas, explain concepts, and chat about a wide range of topics—from science and technology to art, history, and everyday life. Feel free to ask me anything or let me know how I can assist you today!


In [21]:
from typing import Annotated
from langchain_core.tools import InjectedToolArg

@tool
def get_conversion_factor(base_currency:str, target_currency:str) ->float:
  """
  This function fetches the currency conversion factor between the base currency and the target currency
  """
  url = f"https://v6.exchangerate-api.com/v6/YOUR_API_KEY/pair/{base_currency}/{target_currency}"
  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value:int, conversion_rate : Annotated[float,InjectedToolArg]) -> float:
  """
  Given a given currency rate this function calculates the target currency value
  """
  return base_currency_value * conversion_rate



In [22]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])


In [30]:
from langchain.messages import SystemMessage
messages = [
    SystemMessage(content="""
    You are a currency conversion assistant.

    Follow these steps whenever a user requests currency conversion:

    1. Identify the source currency and target currency.
    2. Identify the amount provided by the user.
    3. Call get_conversion_factor to retrieve the exchange rate.
    4. After receiving the exchange rate, call convert.
    5. Pass the amount from the original user request to convert.
    6. The conversion_rate argument is injected by the application.
    7. Do not provide the final answer until the conversion is completed.

    Never invent exchange rates.
    """),

    HumanMessage(content="""
    What is the conversion rate between USD and INR,
    and based on that what will be the value of 10 USD in INR?
    """)
]

In [31]:
messages

[SystemMessage(content='\n    You are a currency conversion assistant.\n\n    Follow these steps whenever a user requests currency conversion:\n\n    1. Identify the source currency and target currency.\n    2. Identify the amount provided by the user.\n    3. Call get_conversion_factor to retrieve the exchange rate.\n    4. After receiving the exchange rate, call convert.\n    5. Pass the amount from the original user request to convert.\n    6. The conversion_rate argument is injected by the application.\n    7. Do not provide the final answer until the conversion is completed.\n\n    Never invent exchange rates.\n    ', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='\n    What is the conversion rate between USD and INR,\n    and based on that what will be the value of 10 USD in INR?\n    ', additional_kwargs={}, response_metadata={})]

In [32]:

import json

conversion_rate = None
max_iterations = 5

for iteration in range(max_iterations):

    # Ask the LLM what to do next
    ai_message = llm_with_tools.invoke(messages)

    messages.append(ai_message)

    # Stop when the LLM has no more tool calls
    if not ai_message.tool_calls:
        print(ai_message.content)
        break

    for tool_call in ai_message.tool_calls:

        if tool_call["name"] == "get_conversion_factor":

            tool_message = get_conversion_factor.invoke(tool_call)

            conversion_rate = json.loads(
                tool_message.content
            )["conversion_rate"]

            messages.append(tool_message)

        elif tool_call["name"] == "convert":

            if conversion_rate is None:
                raise ValueError("Conversion rate not available")

            # Inject the result of the first tool
            tool_call["args"]["conversion_rate"] = conversion_rate

            tool_message = convert.invoke(tool_call)

            messages.append(tool_message)


The current conversion rate from USD to INR is **93.9231**.  
Using that rate, 10 USD equals **939.231 INR**.
